In [1]:
import random
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from PIL import Image
from sklearn.metrics import auc, average_precision_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomRotation, Resize, ToTensor
from tqdm import tqdm

In [2]:
SEED = 492
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
data_path = Path("../../data/ISIC")
image_path = data_path / "ISIC_2024_Training_Input"
ground_truth_path = data_path / "ISIC_2024_Training_GroundTruth.csv"
metadata_path = data_path / "metadata.csv"
model_path = Path("../../models")

In [4]:
device = torch.device("cuda")

In [5]:
ground_truth_df = pl.read_csv(ground_truth_path)
metadata_df = pl.read_csv(metadata_path)

df = ground_truth_df.join(metadata_df, on="isic_id", how="inner").with_columns(
    (pl.lit(str(image_path)) + "/" + pl.col("isic_id").cast(pl.Utf8) + ".jpg").alias("image_path")
)

patient_stats = df.group_by("patient_id").agg(pl.col("malignant").max().alias("has_malignant")).sort("patient_id")

train_p, _ = train_test_split(patient_stats, test_size=0.1, random_state=SEED, stratify=patient_stats["has_malignant"])

train_ids = train_p["patient_id"].to_list()
train_df = df.filter(pl.col("patient_id").is_in(train_ids))

train_malignant = train_df.filter(pl.col("malignant") == 1)
train_benign_all = train_df.filter(pl.col("malignant") == 0)

print(f"Train Malignant: {train_malignant.height}")
print(f"Train Benign (Total Pool): {train_benign_all.height}")

Train Malignant: 348
Train Benign (Total Pool): 352678


In [6]:
tab_categorical = ["sex", "anatom_site_general"]
tab_numerical = ["age_approx", "clin_size_long_diam_mm", "tbp_lv_areaMM2", "tbp_lv_eccentricity"]

capped_benign = (
    train_benign_all.sample(fraction=1.0, shuffle=True, seed=SEED)
    .group_by("patient_id")
    .head(20)
    .select(train_df.columns)
)

final_train_df = pl.concat([train_malignant, capped_benign])

median_age = final_train_df["age_approx"].median()
mode_sex = final_train_df["sex"].drop_nulls().mode()[0]

train_malignant = train_malignant.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown"),
)

train_benign_all = train_benign_all.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown"),
)

final_train_df = final_train_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown"),
)

exprs = []
for c in tab_numerical:
    exprs.append(pl.col(c).mean().alias(f"{c}_mean"))
    exprs.append(pl.col(c).std().alias(f"{c}_std"))

num_stats = final_train_df.select(exprs)

for col in tab_numerical:
    mean = num_stats.item(0, f"{col}_mean")
    std = num_stats.item(0, f"{col}_std")
    train_malignant = train_malignant.with_columns(((pl.col(col) - mean) / std).alias(col))
    train_benign_all = train_benign_all.with_columns(((pl.col(col) - mean) / std).alias(col))

tab_features = list(tab_numerical)

for col in tab_categorical:
    categories = final_train_df[col].unique().to_list()
    for cat in categories:
        col_name = f"{col}_{cat}"
        train_malignant = train_malignant.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(col_name))
        train_benign_all = train_benign_all.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(col_name))
        tab_features.append(col_name)

train_malignant = train_malignant.drop(tab_categorical)
train_benign_all = train_benign_all.drop(tab_categorical)

In [7]:
class ISICInferenceDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame):
        self.df = dataframe
        self.transform = Compose([Resize((224, 224)), ToTensor()])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)
        img = Image.open(row["image_path"]).convert("RGB")
        img = self.transform(img)
        return img, row["isic_id"]

In [8]:
malignant_loader = DataLoader(ISICInferenceDataset(train_malignant), batch_size=128, shuffle=False)
benign_loader = DataLoader(ISICInferenceDataset(train_benign_all), batch_size=128, shuffle=False)

In [9]:
class MultimodalModel(nn.Module):
    def __init__(self, tab_dim):
        super().__init__()
        self.image_encoder = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.image_encoder.fc = nn.Identity()

        self.tab_encoder = nn.Sequential(
            nn.Linear(tab_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
        )

        self.fusion_head = nn.Sequential(
            nn.Linear(2048 + 32, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1),
        )

    def forward(self, image, tab):
        img_feat = self.image_encoder(image)
        tab_feat = self.tab_encoder(tab)
        combined = torch.cat([img_feat, tab_feat], dim=1)
        return self.fusion_head(combined)

In [10]:
class OldMultimodalModel(nn.Module):
    def __init__(self, tab_dim):
        super().__init__()
        self.image_encoder = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.image_encoder.fc = nn.Identity()

        self.tab_encoder = nn.Sequential(nn.Linear(tab_dim, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, 32))

        self.fusion_head = nn.Sequential(nn.Linear(2048 + 32, 512), nn.ReLU(), nn.Dropout(0.1), nn.Linear(512, 1))

    def forward(self, image, tab):
        img_feat = self.image_encoder(image)
        tab_feat = self.tab_encoder(tab)
        combined = torch.cat([img_feat, tab_feat], dim=1)
        return self.fusion_head(combined)


class ImageEmbedder(nn.Module):
    def __init__(self, tab_dim):
        super().__init__()
        full_model = OldMultimodalModel(tab_dim)
        full_model.load_state_dict(torch.load(model_path / "best_multimodal_model.pt", weights_only=True))
        self.encoder = full_model.image_encoder

    def forward(self, x):
        return self.encoder(x)

In [11]:
embedder = ImageEmbedder(tab_dim=len(tab_features)).to(device)
embedder.eval()

ImageEmbedder(
  (encoder): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=

In [12]:
def extract_embeddings(loader, desc):
    embeddings = []
    isic_ids = []
    with torch.no_grad():
        for images, batch_ids in tqdm(loader, desc=desc):
            images = images.to(device)
            with torch.amp.autocast(device_type="cuda"):
                feats = embedder(images)
            embeddings.append(feats.cpu())
            isic_ids.extend(batch_ids)
    return torch.cat(embeddings, dim=0), isic_ids

In [13]:
mal_embs, mal_ids = extract_embeddings(malignant_loader, "Malignant Embeddings")
ben_embs, ben_ids = extract_embeddings(benign_loader, "Benign Embeddings")

Benign Embeddings: 100%|██████████| 2756/2756 [10:46<00:00,  4.27it/s]


In [14]:
distances = torch.cdist(ben_embs, mal_embs, p=2)
min_distances, _ = distances.min(dim=1)

ben_distances_df = pl.DataFrame({"isic_id": ben_ids, "min_mal_dist": min_distances.numpy()})

train_benign_all = train_benign_all.join(ben_distances_df, on="isic_id", how="left")

In [15]:
cols = train_malignant.columns

random_benign_ids = capped_benign["isic_id"].to_list()
random_benign_processed = train_benign_all.filter(pl.col("isic_id").is_in(random_benign_ids)).select(cols)

unused_pool = train_benign_all.filter(~pl.col("isic_id").is_in(random_benign_ids))

hard_benign = unused_pool.sort("min_mal_dist").group_by("patient_id").head(20).select(cols)

mixed_benign = pl.concat([random_benign_processed, hard_benign])
final_train_df_hard = pl.concat([train_malignant, mixed_benign])

print(f"Train Malignant: {train_malignant.height}")
print(f"Train Mixed Benign: {mixed_benign.height}")
print(f"Total Train: {final_train_df_hard.height}")

Train Malignant: 348
Train Mixed Benign: 36099
Total Train: 36447


In [16]:
_, val_p = train_test_split(patient_stats, test_size=0.1, random_state=SEED, stratify=patient_stats["has_malignant"])
val_ids = val_p["patient_id"].to_list()
val_df = df.filter(pl.col("patient_id").is_in(val_ids))

val_df = val_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown"),
)

for col in tab_numerical:
    mean = num_stats.item(0, f"{col}_mean")
    std = num_stats.item(0, f"{col}_std")
    val_df = val_df.with_columns(((pl.col(col) - mean) / std).alias(col))

for col in tab_categorical:
    for feat in tab_features:
        if feat.startswith(f"{col}_"):
            cat = feat[len(col) + 1 :]
            val_df = val_df.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(feat))

val_df = val_df.select(cols)

In [17]:
class ISICMultimodalDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, tabular_features: list[str], image_transform):
        self.df = dataframe
        self.tab_features = tabular_features
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)
        img = Image.open(row["image_path"]).convert("RGB")
        img = self.image_transform(img)
        tab_data = torch.tensor([row[feat] for feat in self.tab_features], dtype=torch.float32)
        label = torch.tensor(row["malignant"], dtype=torch.float32)
        return img, tab_data, label

In [18]:
train_transform = Compose([Resize((224, 224)), RandomHorizontalFlip(p=0.5), RandomRotation(15), ToTensor()])
val_transform = Compose([Resize((224, 224)), ToTensor()])

train_dataset = ISICMultimodalDataset(final_train_df_hard, tab_features, train_transform)
val_dataset = ISICMultimodalDataset(val_df, tab_features, val_transform)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

In [19]:
model = MultimodalModel(tab_dim=len(tab_features)).to(device)

train_malignant_count = final_train_df_hard.filter(pl.col("malignant") == 1).height
train_benign_count = final_train_df_hard.filter(pl.col("malignant") == 0).height

pos_weight = torch.tensor([train_benign_count / train_malignant_count]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=5)
scaler = torch.amp.GradScaler("cuda")

In [20]:
best_ap = 0.0
best_recall_at_spec = 0.0
early_stopping_counter = 0
early_stopping_patience = 9
num_epochs = 30

model_path.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", leave=False)
    for images, tabs, labels in train_loop:
        images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images, tabs)
            loss = criterion(outputs, labels.unsqueeze(1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", leave=False)
    with torch.no_grad():
        for images, tabs, labels in val_loop:
            images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images, tabs)
                loss = criterion(outputs, labels.unsqueeze(1))
            val_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    ap = average_precision_score(all_labels, all_preds)
    fpr, tpr, thresholds = roc_curve(all_labels, all_preds)

    valid_indices = np.where(fpr <= 0.10)[0]
    recall_at_spec = tpr[valid_indices[-1]]

    scheduler.step(ap)

    if ap > best_ap:
        best_ap = ap
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if recall_at_spec > best_recall_at_spec:
        best_recall_at_spec = recall_at_spec
        torch.save(model.state_dict(), model_path / "best_hard_neg_model.pt")

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
        f"AP: {ap:.4f} | Recall@90Spec: {recall_at_spec:.4f}"
    )

    if early_stopping_counter >= early_stopping_patience:
        print(f"Early stopping triggered at epoch {epoch + 1}")
        break

Epoch 1/30 | Train Loss: 1.3470 | Val Loss: 0.6587 | AP: 0.0186 | Recall@90Spec: 0.4222


Epoch 2/30 | Train Loss: 1.2068 | Val Loss: 0.5489 | AP: 0.0583 | Recall@90Spec: 0.6222


Epoch 3/30 | Train Loss: 0.9442 | Val Loss: 0.5267 | AP: 0.0330 | Recall@90Spec: 0.6667


Epoch 4/30 | Train Loss: 0.7490 | Val Loss: 0.4092 | AP: 0.0341 | Recall@90Spec: 0.6889


Epoch 5/30 | Train Loss: 0.6276 | Val Loss: 0.3586 | AP: 0.0353 | Recall@90Spec: 0.7556


...

Epoch 7/30 | Train Loss: 0.5005 | Val Loss: 0.3490 | AP: 0.0275 | Recall@90Spec: 0.8000


Epoch 8/30 | Train Loss: 0.4479 | Val Loss: 0.3068 | AP: 0.0201 | Recall@90Spec: 0.8000


Epoch 9/30 | Train Loss: 0.4025 | Val Loss: 0.3040 | AP: 0.0211 | Recall@90Spec: 0.8222


Epoch 10/30 | Train Loss: 0.3976 | Val Loss: 0.2883 | AP: 0.0200 | Recall@90Spec: 0.8000


Epoch 11/30 | Train Loss: 0.3954 | Val Loss: 0.2991 | AP: 0.0207 | Recall@90Spec: 0.8222
Early stopping triggered at epoch 11


In [21]:
model.load_state_dict(torch.load(model_path / "best_hard_neg_model.pt", weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, tabs, labels in tqdm(val_loader, desc="Evaluating Hard Neg Model"):
        images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images, tabs)
        all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

auc_roc = roc_auc_score(all_labels, all_preds)
ap = average_precision_score(all_labels, all_preds)

fpr, tpr, thresholds = roc_curve(all_labels, all_preds)
valid_idx_90 = np.where(fpr <= 0.10)[0]
threshold_90_spec = thresholds[valid_idx_90[-1]]
recall_90_spec = tpr[valid_idx_90[-1]]

valid_idx_95 = np.where(fpr <= 0.05)[0]
threshold_95_spec = thresholds[valid_idx_95[-1]]
recall_95_spec = tpr[valid_idx_95[-1]]

preds_binary = (all_preds >= threshold_90_spec).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, preds_binary).ravel()

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"Average Precision: {ap:.4f}")
print("--- @ 90% Spec ---")
print(f"Threshold: {threshold_90_spec:.4f} | Recall: {recall_90_spec:.4f}")
print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")
print("--- @ 95% Spec ---")
print(f"Threshold: {threshold_95_spec:.4f} | Recall: {recall_95_spec:.4f}")

Evaluating Hard Neg Model: 100%|██████████| 376/376 [01:19<00:00,  4.74it/s]

AUC-ROC: 0.9079
Average Precision: 0.0211
--- @ 90% Spec ---
Threshold: 0.4719 | Recall: 0.8222
TP: 37 | FP: 4798 | TN: 43190 | FN: 8
--- @ 95% Spec ---
Threshold: 0.7812 | Recall: 0.6222


In [22]:
def p_auc_tpr(v_gt, v_pred, min_tpr=0.80):
    v_gt_flipped = abs(np.asarray(v_gt) - 1)
    v_pred_flipped = abs(np.asarray(v_pred) - 1)
    max_fpr = abs(1 - min_tpr)

    fpr, tpr, _ = roc_curve(v_gt_flipped, v_pred_flipped)

    stop = np.searchsorted(fpr, max_fpr, "right")
    x_interp = [fpr[stop - 1], fpr[stop]]
    y_interp = [tpr[stop - 1], tpr[stop]]

    tpr_adj = np.append(tpr[:stop], np.interp(max_fpr, x_interp, y_interp))
    fpr_adj = np.append(fpr[:stop], max_fpr)

    return auc(fpr_adj, tpr_adj)


isic_pauc = p_auc_tpr(all_labels, all_preds, min_tpr=0.80)
print(f"ISIC 2024 Official Metric (pAUC > 80% TPR): {isic_pauc:.5f}")

ISIC 2024 Official Metric (pAUC > 80% TPR): 0.12931
